In [1]:
import warnings

In [2]:
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
import pandas as pd

In [4]:
from replay.utils.session_handler import State

In [5]:
from replay.utils.spark_utils import convert2spark

In [6]:
from pyspark.sql import SparkSession

In [7]:
session = (
    SparkSession.builder.config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .master("local[*]")
    .enableHiveSupport()
    .getOrCreate()
)
spark = State(session).session

26/05/01 14:34:32 WARN Utils: Your hostname, Antons-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.19 instead (on interface en0)
26/05/01 14:34:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/01 14:34:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/01 14:34:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/01 14:34:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [8]:
spark.sparkContext.setLogLevel("ERROR")

In [9]:
import pickle

In [10]:
with open("artificial_profiles_scores.pkl", "rb") as f:
    scores = pickle.load(f)

In [11]:
df = pd.read_excel("data_clean.xlsx", header=0)

In [12]:
import uuid

In [13]:
data = []
uuid_to_student_map = {}
titles_to_uuid_map = {}
uuid_to_titles_map = {}
for student in scores:
    student_uuid = str(uuid.uuid4())
    uuid_to_student_map[student_uuid] = student
    for title in scores[student]:
        title_uuid = titles_to_uuid_map.get(title)
        if not title_uuid:
            title_uuid = str(uuid.uuid4())
            titles_to_uuid_map[title] = title_uuid
        if title_uuid not in uuid_to_titles_map:
            uuid_to_titles_map[title_uuid] = title
        data.append((student_uuid, student, title_uuid, title, scores[student][title]))
interactions = pd.DataFrame(
    data=data, columns=["user_uuid", "user_name", "item_uuid", "item_name", "rating"]
)

In [14]:
interactions.head()

,user_uuid,user_name,item_uuid,item_name,rating
0,b7d3f0ba-4d76-4a52-80e6-0a4af049076a,international_relations_geopolitics_expert,91b6406f-294b-499a-b646-5e4f59899c03,Монетарная политика и проблема нулевой нижней ...,2
1,b7d3f0ba-4d76-4a52-80e6-0a4af049076a,international_relations_geopolitics_expert,a61be8c9-6dce-4a01-82af-88e26b37c372,Управление проектами и процессами в международ...,2
2,b7d3f0ba-4d76-4a52-80e6-0a4af049076a,international_relations_geopolitics_expert,db82cc60-74f2-40e9-b9c7-87622b6c01ef,VII Международная конференция «Мировое большин...,4
3,b7d3f0ba-4d76-4a52-80e6-0a4af049076a,international_relations_geopolitics_expert,35463c8d-0da6-4917-a55b-3ef480935a7a,Сбор и обработка языкового материала для корпу...,2
4,b7d3f0ba-4d76-4a52-80e6-0a4af049076a,international_relations_geopolitics_expert,af2cd55e-622f-4267-8a1a-03e817698494,Маркетинг в цифровую эпоху,1


In [15]:
interactions.user_uuid.nunique()

200

In [16]:
interactions.item_uuid.nunique()

1187

In [17]:
df = convert2spark(interactions)

In [18]:
from replay.data import Dataset, FeatureHint, FeatureInfo, FeatureSchema, FeatureType

In [19]:
feature_schema = FeatureSchema(
    [
        FeatureInfo(
            column="user_uuid",
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.QUERY_ID,
            cardinality=interactions.user_uuid.nunique(),
        ),
        FeatureInfo(
            column="item_uuid",
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
            cardinality=interactions.item_uuid.nunique(),
        ),
        FeatureInfo(
            column="rating",
            feature_type=FeatureType.NUMERICAL,
            feature_hint=FeatureHint.RATING,
        ),
    ]
)

In [20]:
from replay.splitters import TwoStageSplitter

In [21]:
splitter = TwoStageSplitter(
    query_column="user_uuid",
    item_column="item_uuid",
    first_divide_column="user_uuid",
    second_divide_column="item_uuid",
    drop_cold_items=True,
    drop_cold_users=True,
    first_divide_size=100,
    second_divide_size=100,
    seed=42,
    shuffle=True,
)

In [22]:
train, test = splitter.split(df)

In [23]:
train_dataset = Dataset(
    feature_schema=feature_schema,
    interactions=train,
)

In [24]:
test_dataset = Dataset(
    feature_schema=feature_schema,
    interactions=test,
)

In [25]:
from replay.data.dataset_utils import DatasetLabelEncoder

In [26]:
encoder = DatasetLabelEncoder()
train_dataset = encoder.fit_transform(train_dataset)
test_dataset = encoder.transform(test_dataset)

In [27]:
from replay.models import ALSWrap

/Users/antonshishkov/Projects/diploma/.venv310/lib/python3.10/site-packages/replay/models/optimization/optuna_mixin.py:240: FeatureUnavailableWarning: Optimization feature not enabled - `optuna` package not found. Ensure you have the package installed if you want to use the `optimize()` method in your recommenders.
  warnings.warn(feature_warning)


In [28]:
als = ALSWrap(rank=10, seed=42)

In [29]:
%%time
als.fit(train_dataset)

CPU times: user 29.3 ms, sys: 34.2 ms, total: 63.5 ms
Wall time: 6.34 s


In [30]:
recs = als.predict(
    k=10, queries=test_dataset.query_ids, dataset=train_dataset, filter_seen_items=True
)

In [31]:
# from replay.metrics import Coverage, HitRate, NDCG, MAP, Experiment, OfflineMetrics

In [32]:
# K = 5

In [33]:
# print(
#     NDCG(K, query_column="user_uuid", item_column="item_uuid", rating_column="rating")(recs, test_dataset.interactions),
#     MAP(K, query_column="user_uuid", item_column="item_uuid", rating_column="rating")(recs, test_dataset.interactions),
#     HitRate([1, K], query_column="user_uuid", item_column="item_uuid", rating_column="rating")(recs, test_dataset.interactions),
#     Coverage(K, query_column="user_uuid", item_column="item_uuid", rating_column="rating")(recs, test_dataset.interactions),
# )

In [34]:
recs.show()

+---------+---------+------------------+
|user_uuid|item_uuid|            rating|
+---------+---------+------------------+
|       60|      363|0.8749547600746155|
|       60|      318|0.8600870966911316|
|       60|      740|0.8076444268226624|
|       60|      953|0.8031027913093567|
|       60|      946|0.7971100211143494|
|       60|      837|0.7882260680198669|
|       60|      144|0.7819279432296753|
|       60|     1030|0.7751157283782959|
|       60|      580|0.7663747668266296|
|       60|       24|0.7622048258781433|
|       90|      171|0.6769554018974304|
|       90|       61|0.6541250348091125|
|       90|      853|0.6466203331947327|
|       90|      778|0.6424961686134338|
|       90|      159| 0.641736626625061|
|       90|      590|0.6355007886886597|
|       90|      705| 0.626981258392334|
|       90|     1136|0.6238945722579956|
|       90|      854|0.6214855909347534|
|       90|      273|0.6202276349067688|
+---------+---------+------------------+
only showing top

In [35]:
recs = encoder.query_and_item_id_encoder.inverse_transform(recs)

In [36]:
recs.show()

+------------------+--------------------+--------------------+
|            rating|           user_uuid|           item_uuid|
+------------------+--------------------+--------------------+
|0.7751157283782959|4635f5cd-192f-4ca...|dc7a46c7-0834-434...|
|0.6424961686134338|631a95f3-b760-49b...|a47efc8f-bcbc-411...|
|0.6355007886886597|631a95f3-b760-49b...|7f285de7-7b9d-420...|
|0.6202276349067688|631a95f3-b760-49b...|3aa7df42-81f5-434...|
|0.6769554018974304|631a95f3-b760-49b...|26fbb166-b263-487...|
|0.6238945722579956|631a95f3-b760-49b...|f429523f-b030-494...|
|0.7663747668266296|4635f5cd-192f-4ca...|7d7949eb-dee9-4d0...|
|0.6466203331947327|631a95f3-b760-49b...|b4aecda5-41ad-472...|
|0.7819279432296753|4635f5cd-192f-4ca...|202f9f36-e9ad-49c...|
|0.8076444268226624|4635f5cd-192f-4ca...|9cf8746f-7725-447...|
|0.6214855909347534|631a95f3-b760-49b...|b4cf24c3-aba4-41f...|
|0.7971100211143494|4635f5cd-192f-4ca...|cb012e56-8f91-46e...|
|0.8031027913093567|4635f5cd-192f-4ca...|ccddf448-76d0-

In [37]:
recs_pd = recs.toPandas()

In [38]:
recs_pd["user_name"] = recs_pd.apply(
    lambda x: uuid_to_student_map[x["user_uuid"]], axis=1
)

In [39]:
recs_pd["item_name"] = recs_pd.apply(
    lambda x: uuid_to_titles_map[x["item_uuid"]], axis=1
)

In [40]:
recs_pd.head()

,rating,user_uuid,item_uuid,user_name,item_name
0,0.677908,e7efe008-0d4c-4dd2-9f2e-7199b721b8d3,c32f4029-8fab-4113-85e3-40271350b91e,macroeconomic_geopolitical_analyst,Влияние логистической инфраструктуры e-commerc...
1,0.671743,e7efe008-0d4c-4dd2-9f2e-7199b721b8d3,92332e8d-16be-40e7-bb53-a0a9740fcecf,macroeconomic_geopolitical_analyst,Расщепление фразеологии как прием в поэзии М. ...
2,0.806210,72550bfa-7f71-487e-82a3-98e57610e55e,b9b2b07d-8b0d-4591-8be1-a6b5ab9488ea,cross_cultural_narrative_researcher,Sprachbrücke - Языковой мост
3,0.707173,5f4f1ef0-2b8d-4b18-8c04-de06549f7dcb,ef4f552c-fcfa-4194-943d-50672212876c,social_and_youth_issues_specialist,Медиапродвижение магистерской программы «Мусул...
4,0.831821,3ab47baf-5abe-49b1-940c-801ce025c5d8,b4695387-14ad-41b1-8fcc-9c7c3b44feba,brand_management_and_rebranding_expert,"Цифровой Ботанический сад: структуры, образы, ..."


In [41]:
sorted(recs_pd["user_name"].unique())

['SMM_for_career_and_recruitment',
 'academic_support_and_mentoring',
 'ai_education_and_pedagogy',
 'ai_education_startup_entrepreneur',
 'ai_ethics_law_and_policy',
 'ai_in_business_strategist',
 'applied_language_teacher_specialist',
 'automotive_market_analyst',
 'brand_management_and_rebranding_expert',
 'comparative_law_and_international_relations_specialist',
 'computational_linguistics_and_NLP_enthusiast',
 'content_creator_and_niche_marketing',
 'creative_industries_and_arts_manager',
 'criminal_procedure_and_practice_analyst',
 'cross_cultural_narrative_researcher',
 'cultural_and_media_studies_specialist',
 'cultural_and_social_journalist',
 'cultural_heritage_database_expert',
 'cultural_history_and_geography_expert',
 'cultural_studies_media_strategist',
 'data_and_system_architect',
 'data_storage_and_startup_focus',
 'digital_and_visual_media_creator',
 'econometric_data_scientist',
 'education_and_program_development_manager',
 'education_program_development_specialist'

In [42]:
recs_pd[recs_pd["user_name"] == "nlp_ai_assistant_developer"].sort_values(
    by=["rating"], ascending=False
)

,rating,user_uuid,item_uuid,user_name,item_name
508,0.820212,1e9c30c8-8fd6-41c1-b259-c83696f85958,e9e90eb0-7f4e-4950-a80f-3906fd52ff09,nlp_ai_assistant_developer,Постанархизм и политическая теология: критика ...
96,0.784189,1e9c30c8-8fd6-41c1-b259-c83696f85958,fd7f573d-8fdb-4ca0-9e73-c53ad68eb9af,nlp_ai_assistant_developer,"Правовое просвещение в рамках проекта ""Юридиче..."
173,0.764651,1e9c30c8-8fd6-41c1-b259-c83696f85958,8228ce84-6aa3-4fc3-972b-2729349e112d,nlp_ai_assistant_developer,Специализированные антикоррупционные органы – ...
627,0.758777,1e9c30c8-8fd6-41c1-b259-c83696f85958,ed5af395-04fe-48e3-b0c3-27e0274d3fd4,nlp_ai_assistant_developer,Внешняя политика России в отношении стран Лати...
715,0.755956,1e9c30c8-8fd6-41c1-b259-c83696f85958,6e04e824-f2c3-43c3-9066-4b0f995fc90a,nlp_ai_assistant_developer,Музеи Ирана
341,0.751775,1e9c30c8-8fd6-41c1-b259-c83696f85958,6de04b9b-8e7d-40ff-9c69-45bec36965c3,nlp_ai_assistant_developer,Digital humanities в действии: кейс-стади на м...
266,0.746286,1e9c30c8-8fd6-41c1-b259-c83696f85958,14c094ca-8e3f-4d12-83e0-7b7d3ea221c9,nlp_ai_assistant_developer,Государственный суверенитет в условиях глобаль...
174,0.746101,1e9c30c8-8fd6-41c1-b259-c83696f85958,09b2b15b-7d04-46eb-bb27-1183dff7941b,nlp_ai_assistant_developer,Шедевры и санкции: спрос на киноискусство в Ро...
267,0.746099,1e9c30c8-8fd6-41c1-b259-c83696f85958,d3598460-9e0f-4f0b-9a10-563f1b99dd0b,nlp_ai_assistant_developer,Практика жанра
509,0.744555,1e9c30c8-8fd6-41c1-b259-c83696f85958,bfb2a0d8-6a90-4344-a992-0feca7165f4d,nlp_ai_assistant_developer,Использование искусственного интеллекта для оп...


In [43]:
recs_pd[recs_pd["user_name"] == "conference_and_forum_organizer"].sort_values(
    by=["rating"], ascending=False
)

,rating,user_uuid,item_uuid,user_name,item_name


In [44]:
recs_pd[recs_pd["user_name"] == "sports_event_coordinator"].sort_values(
    by=["rating"], ascending=False
)

,rating,user_uuid,item_uuid,user_name,item_name
971,0.852867,6565e47b-903c-4c5e-89c0-352bec72ced5,4fcd6bf4-16f3-4c1c-8e3c-b66ab6a080cd,sports_event_coordinator,Архивариус
887,0.839700,6565e47b-903c-4c5e-89c0-352bec72ced5,59ba73a5-2833-4e20-a975-cb40c5c4d95e,sports_event_coordinator,Организация и проведение мероприятий для абиту...
678,0.837555,6565e47b-903c-4c5e-89c0-352bec72ced5,b20caf8a-0e09-4477-842e-3a26abbf2bb7,sports_event_coordinator,Стажировка в редакции журнала «Vox medii aevi»
801,0.805455,6565e47b-903c-4c5e-89c0-352bec72ced5,074cbfcb-7978-46d7-855b-62088f1040d1,sports_event_coordinator,Разработка стратегической и контентно-коммуник...
519,0.799920,6565e47b-903c-4c5e-89c0-352bec72ced5,6a66ac54-1afe-462e-9802-518321b8c2a4,sports_event_coordinator,Динамика социальных детерминантов здоровья дет...
926,0.799498,6565e47b-903c-4c5e-89c0-352bec72ced5,0096740e-de71-430d-b785-7cef7db494ce,sports_event_coordinator,Анализ метамотивационных оснований существован...
42,0.798992,6565e47b-903c-4c5e-89c0-352bec72ced5,d95c4df8-9a6a-431e-9b51-b87d872a8eb3,sports_event_coordinator,Фестиваль языков и культур: подготовка и прове...
241,0.798771,6565e47b-903c-4c5e-89c0-352bec72ced5,cbcacad5-e31b-4c89-a83b-cbedddeaacbc,sports_event_coordinator,"Онлайн-журнал ""MindMasters"""
242,0.791872,6565e47b-903c-4c5e-89c0-352bec72ced5,a5469624-61e9-4c29-a31e-c9b3a54fb1a0,sports_event_coordinator,Анализ курсов по цифровым инструментам и искус...
836,0.790998,6565e47b-903c-4c5e-89c0-352bec72ced5,167b1e28-d5df-451a-82c9-2be4bbda7e96,sports_event_coordinator,Влияние санкционных ограничений на финансовые ...


In [45]:
recs_pd[
    recs_pd["user_name"] == "financial_regulator_and_compliance_expert"
].sort_values(by=["rating"], ascending=False)

,rating,user_uuid,item_uuid,user_name,item_name
